#### Versuch exact matching zu evaluieren

In [8]:
import os
import re
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

def read_file(file_path):
    with open(file_path, 'r', encoding="utf-8") as file:
        return file.read()

def extract_operators(text):
    operators = re.findall(r'\[(AND|OR|NOT)\]', text)
    return operators

def find_operator_words(text, operators):
    words = {op: [] for op in operators}
    word_list = text.split()
    for i, word in enumerate(word_list):
        if word in ["[AND]", "[OR]", "[NOT]"]:
            operator = word[1:-1]
            if operator == 'NOT' and i + 1 < len(word_list):
                words[operator].append(word_list[i + 1])
            elif operator in ['AND', 'OR'] and i - 1 >= 0:
                prev_word = re.sub(r'[^\w]', '', word_list[i - 1])  # Entferne Satzzeichen
                words[operator].append(prev_word)
    return words

def compare_operators(label_text, model_text, operator):
    label_operators = extract_operators(label_text)
    model_operators = extract_operators(model_text)

    label_words_dict = find_operator_words(label_text, label_operators)
    model_words_dict = find_operator_words(model_text, model_operators)

    label_words = label_words_dict.get(operator, [])
    model_words = model_words_dict.get(operator, [])

    tp = sum(1 for word in label_words if word in model_words)
    fp = sum(1 for word in model_words if word not in label_words)
    fn = sum(1 for word in label_words if word not in model_words)

    return tp, fp, fn, len(label_words), len(model_words)

def calculate_metrics(tp, fp, fn):
    precision = tp / (tp + fp) if tp + fp > 0 else 0
    recall = tp / (tp + fn) if tp + fn > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if precision + recall > 0 else 0
    accuracy = tp / (tp + fn) if tp + fn > 0 else 0
    return precision, recall, f1, accuracy

def evaluate_models(label_folder, model_folder):
    operators = ['AND', 'OR', 'NOT']
    metrics = {op: {'tp': 0, 'fp': 0, 'fn': 0, 'label_count': 0, 'model_count': 0, 'correct_count': 0, 'correct_ncts': []} for op in operators}
    processed_label_files = 0

    for model_file in os.listdir(model_folder):
        if model_file.endswith('.txt'):
            nct_number = extract_nct_number(model_file)

            if nct_number:
                label_file_path = os.path.join(label_folder, f'{nct_number}.txt')
                model_file_path = os.path.join(model_folder, model_file)

                if os.path.exists(label_file_path):
                    processed_label_files += 1
                    label_text = read_file(label_file_path)
                    model_text = read_file(model_file_path)

                    for op in operators:
                        tp, fp, fn, label_count, model_count = compare_operators(label_text, model_text, op)

                        metrics[op]['tp'] += tp
                        metrics[op]['fp'] += fp
                        metrics[op]['fn'] += fn
                        metrics[op]['label_count'] += label_count
                        metrics[op]['model_count'] += model_count
                        metrics[op]['correct_count'] += tp

                        if tp > 0:
                            metrics[op]['correct_ncts'].append(nct_number)

    print(f"Total processed label files: {processed_label_files}")
    for op in operators:
        precision, recall, f1, accuracy = calculate_metrics(metrics[op]['tp'], metrics[op]['fp'], metrics[op]['fn'])
        print(f"{op}:")
        print(f"  Precision: {precision:.3f}")
        print(f"  Recall: {recall:.3f}")
        print(f"  F1-score: {f1:.3f}")
        print(f"  Accuracy: {accuracy:.3f}")
        print(f"  Total in Label: {metrics[op]['label_count']}")
        print(f"  Total in Model: {metrics[op]['model_count']}")
        print(f"  Correctly Identified: {metrics[op]['correct_count']}")
        #print(f"  Correct NCTs: {metrics[op]['correct_ncts']}\n")

def extract_nct_number(filename):
    match = re.search(r'NCT\d+', filename)
    return match.group() if match else None

In [9]:
label_folder = '../../input/lct_p1'
model_folder = 'model_output/Llama-3-70B-Instruct_4_shot/output'

evaluate_models(label_folder, model_folder)

Total processed label files: 1006
AND:
  Precision: 0.150
  Recall: 0.266
  F1-score: 0.192
  Accuracy: 0.266
  Total in Label: 819
  Total in Model: 1454
  Correctly Identified: 218
OR:
  Precision: 0.683
  Recall: 0.562
  F1-score: 0.617
  Accuracy: 0.562
  Total in Label: 4153
  Total in Model: 3380
  Correctly Identified: 2332
NOT:
  Precision: 0.027
  Recall: 0.013
  F1-score: 0.018
  Accuracy: 0.013
  Total in Label: 922
  Total in Model: 441
  Correctly Identified: 12


In [10]:
label_folder = '../../input/lct_p1'
model_folder = 'model_output/Llama-3-8B-Instruct_4_shot/output'
evaluate_models(label_folder, model_folder)

Total processed label files: 1006
AND:
  Precision: 0.121
  Recall: 0.155
  F1-score: 0.136
  Accuracy: 0.155
  Total in Label: 819
  Total in Model: 1051
  Correctly Identified: 127
OR:
  Precision: 0.644
  Recall: 0.241
  F1-score: 0.351
  Accuracy: 0.241
  Total in Label: 4153
  Total in Model: 1517
  Correctly Identified: 1002
NOT:
  Precision: 0.061
  Recall: 0.067
  F1-score: 0.064
  Accuracy: 0.067
  Total in Label: 922
  Total in Model: 1014
  Correctly Identified: 62


In [ ]:
## Baue 5 shot auf 
# NCT03865433.txt
# NCT03860324.txt
# NCT03860233.txt
# NCT03923231.txt
# NCT03930121.txt

In [ ]:
shot_list = [
    "NCT03865433.txt",
    "NCT03860324.txt",
    "NCT03860233.txt",
    "NCT03923231.txt",
    "NCT03930121.txt"
]